In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb

In [ ]:
def add_features(data):
    data = data.copy()
    data['Sample Date'] = pd.to_datetime(data['Sample Date'], format='%d-%m-%Y', errors='coerce')
    data = data.dropna(subset=['Sample Date'])
    
    data['lat'] = data['Latitude']
    data['lon'] = data['Longitude']
    
    for angle, prefix in [(data['lat'], 'lat'), (data['lon'], 'lon')]:
        rad = np.radians(angle)
        for k in [1, 2, 3, 4]:
            data[f'{prefix}_sin_{k}'] = np.sin(k * rad)
            data[f'{prefix}_cos_{k}'] = np.cos(k * rad)
    
    data['year'] = data['Sample Date'].dt.year.astype(float)
    data['year_norm'] = (data['year'] - 2011) / 4.0
    data['month'] = data['Sample Date'].dt.month.astype(float)
    data['doy']   = data['Sample Date'].dt.dayofyear.astype(float)
    
    data['doy_sin_365'] = np.sin(2 * np.pi * data['doy'] / 365.25)
    data['doy_cos_365'] = np.cos(2 * np.pi * data['doy'] / 365.25)
    data['month_sin_12'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos_12'] = np.cos(2 * np.pi * data['month'] / 12)
    
    for period, label in [(182.625, '182_6'), (91.3125, '91_3'), (30.4375, '30_4')]:
        data[f'doy_sin_{label}'] = np.sin(2 * np.pi * data['doy'] / period)
        data[f'doy_cos_{label}'] = np.cos(2 * np.pi * data['doy'] / period)
    
    data['lat_doy_sin'] = data['lat_sin_1'] * data['doy_sin_365']
    data['lon_doy_sin'] = data['lon_sin_1'] * data['doy_sin_365']
    data['lat_year']    = data['lat'] * data['year_norm']
    data['lon_year']    = data['lon'] * data['year_norm']
    
    return data

In [4]:
print("=== Loading & merging larger training data ===")

df_target = pd.read_csv('water_quality_training_dataset.csv')

df_terra = pd.read_csv('terraclimate_features_training.csv')

df_landsat1 = pd.read_csv('landsat_features_training.csv')
df_landsat2 = pd.read_csv('landsat_features_2000_2006_200_samples.csv')
df_landsat3 = pd.read_csv('landsat_features_2015_2023_200_samples.csv')
df_landsat = pd.concat([df_landsat1, df_landsat2, df_landsat3], ignore_index=True)

df_deltares = pd.read_csv('deltares_water_2010_2015_200_samples.csv')
df_train = df_target.merge(df_terra, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_landsat, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_deltares, on=['Latitude', 'Longitude', 'Sample Date'], how='left')

df_train = add_features(df_train)

=== Loading & merging larger training data ===


In [5]:
raw_features = ['pet', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
                'P', 'ETa', 'PET', 'Melt', 'Snow', 'Temp', 'P_res', 'S_res', 
                'Ea_res', 'Qin_res', 'FracFull', 'Qout_res']

engineered = [c for c in df_train.columns if any(x in c for x in [
    'sin', 'cos', 'year', 'doy_', 'month_', 'lat_', 'lon_'
])]

features = raw_features + engineered

print(f"Using {len(features)} features (including extra Landsat & Deltares samples)")

Using 51 features (including extra Landsat & Deltares samples)


In [6]:
X = df_train[features].fillna(0)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

targets = {
    'TA':  df_train['Total Alkalinity'],
    'EC':  df_train['Electrical Conductance'],
    'DRP': df_train['Dissolved Reactive Phosphorus']
}

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.012,
    'max_depth': 6,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'tree_method': 'hist',
    'seed': 42  
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {}

for name, y in targets.items():
    print(f"\n=== Training {name} ===")
    oof = np.zeros(len(y))
    model_list = []
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_scaled)):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        evals = [(dval, 'eval')]
        
        booster = xgb.train(
            xgb_params,
            dtrain,
            num_boost_round=1200,
            evals=evals,
            early_stopping_rounds=60,
            verbose_eval=100
        )
        
        model_list.append(booster)
        
        oof[val_idx] = booster.predict(dval)
    
    r2 = r2_score(y, oof)
    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"{name}  CV R² = {r2:.4f}   RMSE = {rmse:.2f}")
    
    models[name] = model_list

In [ ]:
print("\n=== Generating submission ===")
sub = pd.read_csv('submission_template.csv')

df_terra_val = pd.read_csv('terraclimate_features_validation.csv')
df_landsat_val = pd.read_csv('landsat_features_validation.csv')

sub = sub.merge(df_terra_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub = sub.merge(df_landsat_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub = add_features(sub)

deltares_cols = ['P', 'ETa', 'PET', 'Melt', 'Snow', 'Temp', 'P_res', 'S_res', 
                 'Ea_res', 'Qin_res', 'FracFull', 'Qout_res']
for col in deltares_cols:
    if col not in sub.columns:
        sub[col] = 0

sub_X = sub[features].fillna(0)
sub_X_scaled = scaler.transform(sub_X)

def ensemble_predict(models_list, X):
    dsub = xgb.DMatrix(X)
    return np.mean([m.predict(dsub) for m in models_list], axis=0)

sub['Total Alkalinity']              = ensemble_predict(models['TA'],  sub_X_scaled)
sub['Electrical Conductance']        = ensemble_predict(models['EC'],  sub_X_scaled)
sub['Dissolved Reactive Phosphorus'] = ensemble_predict(models['DRP'], sub_X_scaled)

# Clipping (same as your best model)
sub['Total Alkalinity']              = sub['Total Alkalinity'].clip(5, 400)
sub['Electrical Conductance']        = sub['Electrical Conductance'].clip(30, 2200)
sub['Dissolved Reactive Phosphorus'] = sub['Dissolved Reactive Phosphorus'].clip(0.5, 250)

final_sub = sub[['Latitude', 'Longitude', 'Sample Date',
                 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

final_sub.to_csv('submission_larger_landsat_deltares_xgb.csv', index=False)
print("✅ Saved → submission_larger_landsat_deltares_xgb.csv")
print("Upload this file and check the leaderboard!")